In [ ]:
# ============================================================
# Block 1: Compute Daily NAM (Monthly EOF + DOY Projection)
# ============================================================

import os
import re
import glob
import warnings
import numpy as np
import xarray as xr
import dask
from dask.diagnostics import ProgressBar

warnings.filterwarnings("ignore")
dask.config.set(scheduler="threads", num_workers=8)

# -----------------------------
# 1. 路径与配置 (请确认路径无误)
# -----------------------------
DATA_DIR = "/pool/datos/reanalisis/era5/6hourly/global"
OUT_DIR  = "/home/weiji/restart_exam/code/20260315NAM/resul/New_NAM_Algorithm"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_NAM_NC = os.path.join(OUT_DIR, "ERA5_daily_vertical_NAM_monthlyEOF_DOYprojected.nc")

G = 9.80665
YEAR_MIN, YEAR_MAX = 1950, 2020
REF_START, REF_END = "1950-01-01", "2019-12-31"
LAT_MIN_EOF, LAT_MAX_EOF = 20.0, 90.0

def preprocess(ds):
    return ds[["z"]].sel(latitude=slice(LAT_MAX_EOF, LAT_MIN_EOF))

# -----------------------------
# 2. 读取数据并计算日均/月均纬向平均
# -----------------------------
print("⏳ 1. 加载 ERA5 数据并计算纬向平均 (Zonal Mean)...")
pattern = re.compile(r"^z_4daily_(\d{8})-(\d{8})\.nc$")
z_files = [f for f in sorted(glob.glob(os.path.join(DATA_DIR, "z_4daily*.nc"))) 
           if pattern.match(os.path.basename(f)) and YEAR_MIN <= int(pattern.match(os.path.basename(f)).group(1)[:4]) <= YEAR_MAX]

ds = xr.open_mfdataset(z_files, combine="by_coords", parallel=True, preprocess=preprocess, chunks={"time": 180})
zg_4x_zm = (ds["z"] / G).mean("longitude")

with ProgressBar():
    zg_daily_zm = zg_4x_zm.resample(time="1D").mean().astype("float32").load()
ds.close()

zg_monthly_zm = zg_daily_zm.resample(time="MS").mean()

# -----------------------------
# 3. 计算参考态与异常场 (核心逻辑修正)
# -----------------------------
print("⏳ 2. 计算 Monthly 气候态(用于 EOF) 与 DOY 气候态(用于日投影)...")

# A. 用于提取 EOF 的月度异常 (严格参考 Samuel)
ref_monthly = zg_monthly_zm.sel(time=slice(REF_START, REF_END))
clim_mon = ref_monthly.groupby("time.month").mean("time")
anom_monthly_ref = ref_monthly.groupby("time.month") - clim_mon

# B. 用于每日投影的 DOY 异常 (消除跨月跳变)
ref_daily = zg_daily_zm.sel(time=slice(REF_START, REF_END))
clim_doy = ref_daily.groupby("time.dayofyear").mean("time")
# 加上平滑让背景态绝对丝滑 (可选，但推荐)
clim_doy_smooth = clim_doy.rolling(dayofyear=21, center=True).mean().bfill("dayofyear").ffill("dayofyear")

anom_daily_all = zg_daily_zm.groupby("time.dayofyear") - clim_doy_smooth
if "dayofyear" in anom_daily_all.coords:
    anom_daily_all = anom_daily_all.drop_vars("dayofyear")

# -----------------------------
# 4. 逐层计算 EOF1 并进行 DOY 投影
# -----------------------------
print("⏳ 3. 逐层计算 EOF1 并投影...")
levels = anom_monthly_ref["level"].values
lat_vals = anom_monthly_ref["latitude"].values
times_all = anom_daily_all["time"].values

# 面积加权 sqrt(cos(lat))
coslat = np.clip(np.cos(np.deg2rad(lat_vals.astype(np.float64))), 0.0, None)
weights = np.sqrt(coslat)

Xref_month = anom_monthly_ref.transpose("level", "latitude", "time").values.astype(np.float64)
Xday_all   = anom_daily_all.transpose("level", "latitude", "time").values.astype(np.float64)

eof_daily_nam = np.full((len(times_all), len(levels)), np.nan, dtype=np.float64)

for ilev, lev in enumerate(levels):
    # 加权
    Xref = Xref_month[ilev, :, :] * weights[:, None]
    Xday = Xday_all[ilev, :, :] * weights[:, None]
    
    # 过滤 NaN
    valid_mon_t = np.all(np.isfinite(Xref), axis=0)
    Xref_valid = Xref[:, valid_mon_t]
    
    # 求解 EOF
    train_mean = Xref_valid.mean(axis=1, keepdims=True)
    Xc = Xref_valid - train_mean
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    eof1 = U[:, 0]
    
    # Samuel 符号约定：强制 80N 附近为负
    i80 = int(np.abs(lat_vals - 80.0).argmin())
    if eof1[i80] > 0:
        eof1 = -eof1
        
    # 计算参考期 PC 用于后续标准化
    pc_ref = eof1 @ Xc
    pc_ref_mean = np.mean(pc_ref)
    pc_ref_std = np.std(pc_ref, ddof=1)
    
    # DOY 日数据投影
    valid_day_t = np.all(np.isfinite(Xday), axis=0)
    if valid_day_t.sum() > 0:
        Xday_valid = Xday[:, valid_day_t]
        Xday_c = Xday_valid - train_mean
        pc_day = eof1 @ Xday_c
        
        # 标准化
        nam_day_valid = (pc_day - pc_ref_mean) / pc_ref_std
        eof_daily_nam[valid_day_t, ilev] = nam_day_valid

# -----------------------------
# 5. 保存结果
# -----------------------------
namEOF_daily = xr.DataArray(
    eof_daily_nam,
    coords={"time": times_all, "level": levels},
    dims=("time", "level"),
    name="NAM"
)
namEOF_daily.attrs["long_name"] = "Daily NAM (Monthly EOF1 projected on DOY anomalies)"
namEOF_daily.to_dataset(name="NAM").to_netcdf(OUT_NAM_NC)
print(f"✅ NAM 计算完成！文件已保存至: {OUT_NAM_NC}")

In [ ]:
# ============================================================
# Block 2: Plot Figure 3a,b (New NAM + Pre-computed ERA5 AO)
# ============================================================

import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1. 路径与读取配置
# -----------------------------
NAM_FILE = "/home/weiji/restart_exam/code/20260315NAM/resul/New_NAM_Algorithm/ERA5_daily_vertical_NAM_monthlyEOF_DOYprojected.nc"

# 👇 请将这里替换为你真实存在的 AO_ERA5 文件路径
ERA5_AO_FILE = "/home/weiji/restart_exam/code/20260324_AO_compare_A_vs_Bstyle_STRICT/AO_codeA.nc" 
OUT_FIG = "/home/weiji/restart_exam/code/20260315NAM/resul/New_NAM_Algorithm/Figure3ab_Final_NAM_AO.png"

START_DATE = "2019-11-01"
END_DATE   = "2020-05-31"

print("📂 1. 读取 NAM 与 ERA5 AO 数据...")

# 读取 NAM
ds_nam = xr.open_dataset(NAM_FILE)
nam_sub = ds_nam["NAM"].sel(time=slice(START_DATE, END_DATE))

# 读取 AO (兼容 CSV 或 NC)
if ERA5_AO_FILE.endswith(".csv"):
    ao_df = pd.read_csv(ERA5_AO_FILE, parse_dates=["time"])
else:
    ds_ao = xr.open_dataset(ERA5_AO_FILE)
    var_name = list(ds_ao.data_vars)[0]
    ao_df = ds_ao[var_name].to_dataframe().reset_index()

# 对齐时间窗
ao_sub = ao_df[(ao_df["time"] >= START_DATE) & (ao_df["time"] <= END_DATE)].copy()
ao_val_col = [c for c in ao_sub.columns if c != "time" and pd.api.types.is_numeric_dtype(ao_sub[c])][0]

ao_time = ao_sub["time"].values
ao_vals = ao_sub[ao_val_col].values

# -----------------------------
# 2. 绘图 (Figure 3a,b)
# -----------------------------
print("🎨 2. 绘制组合图...")
fig = plt.figure(figsize=(10, 6))
gs = fig.add_gridspec(nrows=2, ncols=2, height_ratios=[2.5, 1.0], width_ratios=[1, 0.02], hspace=0.1, wspace=0.02)

# ---- (a) NAM time-height ----
ax1 = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1])

levels_c = np.arange(-4.5, 5.0, 0.5)
cf = nam_sub.plot.contourf(
    ax=ax1, x="time", y="level", levels=levels_c, 
    cmap="RdBu", extend="both", add_colorbar=False
)

ax1.set_yscale("log")
ax1.invert_yaxis()
ax1.set_ylim(1000, 1)
ax1.set_yticks([1000, 300, 100, 30, 10, 3, 1])
ax1.get_yaxis().set_major_formatter(plt.ScalarFormatter())
ax1.set_ylabel("Pressure [hPa]", fontsize=15)
ax1.set_xlabel("")
ax1.tick_params(labelbottom=False)
ax1.set_title("")
ax1.text(0.01, 0.92, "(a)", transform=ax1.transAxes, fontsize=15, fontweight="bold", bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=2))

cbar = fig.colorbar(cf, cax=cax)
cbar.set_label("NAM Index", fontsize=13)

# ---- (b) ERA5 AO ----
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
ax2.plot(ao_time, ao_vals, color="black", lw=2)
ax2.axhline(0, color="gray", lw=1.2, linestyle="-")

ax2.set_ylabel(r"$AO_{\mathrm{ERA5}}$", fontsize=15)
ax2.set_ylim(-4.5, 4.5)
ax2.set_yticks([-3, 0, 3])
ax2.text(0.01, 0.82, "(b)", transform=ax2.transAxes, fontsize=15, fontweight="bold", bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=2))
ax2.grid(True, linestyle=":", alpha=0.6)

fig.suptitle("Daily NAM (Monthly EOF projected) and ERA5 AO", fontsize=16, fontweight="bold", y=0.93)
plt.savefig(OUT_FIG, dpi=300, bbox_inches="tight")
print(f"✅ 图表已保存至: {OUT_FIG}")

plt.show()
ds_nam.close()

In [ ]:
# ============================================================
# Block 3: Debugging - Plot line charts for selected pressure levels
# ============================================================

import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# -----------------------------
# 路径配置 (读取你在 Block 1 刚算出来的 NAM 文件)
# -----------------------------
NAM_FILE = "/home/weiji/restart_exam/code/20260315NAM/resul/New_NAM_Algorithm/ERA5_daily_vertical_NAM_monthlyEOF_projected.nc"

START_DATE = "2019-11-01"
END_DATE   = "2020-05-31"

print("📂 1. 读取 NAM 数据...")
ds_nam = xr.open_dataset(NAM_FILE)

# 截取需要查看的时间段
nam_sub = ds_nam["NAM"].sel(time=slice(START_DATE, END_DATE))
available_levels = ds_nam["level"].values

# 用户想要查看的目标气压层
target_levels = [1, 10, 50, 100, 500, 800]
actual_levels = []

print("\n🔍 2. 指定气压层统计信息诊断:")
print("-" * 80)
for target in target_levels:
    # 找到最接近的目标层 (比如 ERA5 可能有 850hPa 却没有 800hPa)
    closest_lev = available_levels[np.abs(available_levels - target).argmin()]
    actual_levels.append(closest_lev)
    
    # 获取该层在这半年内的数值
    data_lev = nam_sub.sel(level=closest_lev).values
    
    # 计算统计特征
    n_nan = np.isnan(data_lev).sum()
    mean_val = np.nanmean(data_lev)
    std_val = np.nanstd(data_lev)
    min_val = np.nanmin(data_lev)
    max_val = np.nanmax(data_lev)
    
    print(f"目标: {target:>3} hPa -> 实际提取: {closest_lev:>3} hPa | "
          f"NaN数: {n_nan:>3} | 均值: {mean_val:>6.3f} | "
          f"方差: {std_val:>5.3f} | 最小: {min_val:>6.3f} | 最大: {max_val:>6.3f}")
print("-" * 80)

# -----------------------------
# 3. 绘制多层折线图比较
# -----------------------------
print("\n🎨 3. 正在绘制时间序列折线图...")
fig, ax = plt.subplots(figsize=(12, 6))

time_vals = nam_sub["time"].values
colors = plt.cm.tab10(np.linspace(0, 1, len(actual_levels)))

for i, lev in enumerate(actual_levels):
    data_lev = nam_sub.sel(level=lev).values
    ax.plot(time_vals, data_lev, label=f"{lev} hPa", color=colors[i], linewidth=2.0, alpha=0.85)

# 画一条水平 0 线
ax.axhline(0, color='black', linestyle='--', linewidth=1.5, alpha=0.6)

# 图表设置
ax.set_title(f"NAM Index at Selected Pressure Levels ({START_DATE} to {END_DATE})", fontsize=15, fontweight='bold', pad=15)
ax.set_ylabel("NAM Index (Standardized)", fontsize=13)
ax.set_xlabel("Date", fontsize=13)
ax.grid(True, linestyle=':', alpha=0.7)

# 把图例放右边外面，免得遮挡线
ax.legend(title="Pressure Level", fontsize=11, title_fontsize=12, loc='center left', bbox_to_anchor=(1.02, 0.5))

plt.tight_layout()

# 保存诊断图
OUT_FIG_DEBUG = "/home/weiji/restart_exam/code/20260315NAM/resul/New_NAM_Algorithm/Debug_NAM_linecharts.png"
fig.savefig(OUT_FIG_DEBUG, dpi=200, bbox_inches="tight")
print(f"✅ 诊断图已保存至: {OUT_FIG_DEBUG}")

plt.show()

# 记得关闭数据释放内存
ds_nam.close()

In [ ]:
# ============================================================
# Block 4: Diagnostic - Find Max Daily Jump in NAM
# ============================================================

import xarray as xr
import numpy as np
import pandas as pd

NAM_FILE = "/home/weiji/restart_exam/code/20260315NAM/resul/New_NAM_Algorithm/ERA5_daily_vertical_NAM_monthlyEOF_projected.nc"
START_DATE = "2019-11-01"
END_DATE   = "2020-05-31"

print("🔍 分析 NAM 相邻时间步长的跳变情况...\n")

ds_nam = xr.open_dataset(NAM_FILE)
nam_sub = ds_nam["NAM"].sel(time=slice(START_DATE, END_DATE))

available_levels = ds_nam["level"].values
target_levels = [1, 10, 50, 100, 500, 800]
time_vals = nam_sub["time"].values

print("-" * 70)
print(f"{'Level':>8} | {'Max Jump (abs)':>15} | {'Date of Max Jump':>20} | {'Values (Day 1 -> Day 2)':>20}")
print("-" * 70)

for target in target_levels:
    # 找最近的层
    closest_lev = available_levels[np.abs(available_levels - target).argmin()]
    
    # 提取该层数据
    data_lev = nam_sub.sel(level=closest_lev).values
    
    # 计算相邻两天的差值（绝对值）
    # np.diff 计算的是 a[n+1] - a[n]
    diffs = np.diff(data_lev)
    abs_diffs = np.abs(diffs)
    
    # 找到最大跳变的位置
    max_idx = np.nanargmax(abs_diffs)
    max_jump = abs_diffs[max_idx]
    
    # 获取发生跳变的两天日期和数值
    date1 = pd.to_datetime(time_vals[max_idx]).strftime('%Y-%m-%d')
    date2 = pd.to_datetime(time_vals[max_idx + 1]).strftime('%Y-%m-%d')
    val1 = data_lev[max_idx]
    val2 = data_lev[max_idx + 1]
    
    print(f"{closest_lev:>4} hPa | {max_jump:>15.3f} | {date1} -> {date2} | {val1:>7.2f} -> {val2:>7.2f}")

print("-" * 70)

# ============================================================
# 额外检查：高频反转检查
# 如果数据像锯齿一样天天翻转，那么 diff 的正负号会频繁改变
# ============================================================
print("\n🔍 高频震荡检查：")
lev_10 = available_levels[np.abs(available_levels - 10).argmin()]
data_10 = nam_sub.sel(level=lev_10).values
diffs_10 = np.diff(data_10)

# 计算符号改变的次数
sign_changes = np.sum(np.sign(diffs_10[:-1]) != np.sign(diffs_10[1:]))
total_steps = len(diffs_10) - 1

print(f"在 10 hPa 处，总共有 {total_steps} 个步伐，其中有 {sign_changes} 次步伐方向发生翻转。")
if sign_changes / total_steps > 0.4:
    print("⚠️ 警告：极度频繁的震荡！这说明并不是偶尔的跳跃，而是天天在锯齿状跳动。")
    print("这通常是因为在投影 (Projection) 时，日异常场 (anom_daily) 计算错误，或者被减去的月平均气候态由于某种原因(比如groupby的错位)导致每天都在减去截然不同的值。")
else:
    print("震荡频率正常，可能是特定日期的极端天气事件或个别数据异常。")

ds_nam.close()